In [ ]:
!pip install -U langchain langchain_openai langchain_qdrant qdrant-client

In [5]:
from langchain.agents import create_agent
from langchain.messages import SystemMessage, HumanMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain.tools import tool
import os
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [2]:
model = "gpt-4o-mini"
llm_model = ChatOpenAI(
    base_url="https://api.openai.com/v1",
    openai_proxy=None,
    model = model,
    stream_usage=True,
    streaming=True,
    api_key = os.getenv("OPENAI_API_KEY")
)

In [ ]:
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from langchain_openai import OpenAIEmbeddings

# Cargar credenciales desde variables de entorno
qdrant_url = os.getenv("QDRANT_URL")
qdrant_key = os.getenv("QDRANT_API_KEY")
collection_name = os.getenv("QDRANT_COLLECTION_NAME", "academy")

# Configure embeddings (usa el mismo modelo que el notebook de ingesta)
embeddings_model = "text-embedding-3-small"
embeddings = OpenAIEmbeddings(model=embeddings_model)

client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_key
)

# Crear el vector store en Qdrant
qdrant_store = QdrantVectorStore(
    client=client,
    collection_name=collection_name,
    embedding=embeddings
)

In [7]:
from datetime import datetime

@tool(response_format="content_and_artifact")
def retrieve_context(query: str):
    """Recupera informacion para ayudar a responder informacion sobre matriculas."""
    retrieved_docs = qdrant_store.similarity_search(query, k=5)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\nContent: {doc.page_content}")
        for doc in retrieved_docs
    )
    return serialized, retrieved_docs

@tool
def get_weather(location: str) -> str:
    """Get weather information for a location."""
    return f"Weather in {location}: Sunny, 72°F"

@tool
def sumar(a: int, b: int) -> int:
  """
  Dados dos parametros a y b, devuelve la suma de ambos

  @param a: primer numero
  @param b: segundo numero
  @return: la suma de a y b
  """
  print(f"sumando a: {a} con b: {b}")
  return a + b

@tool
def multiplicar(a: int, b: int) -> int:
  """
  Dados dos parametros a y b, devuelve la multiplicacion de ambos
  @param a: primer numero
  @param b: segundo numero
  @return: la multiplicacion de a y b
  """
  print(f"multiplicando a: {a} con b: {b}")
  return a * b

@tool
def restar(a: int, b: int) -> int:
  """
  Dads dos parametros a y b, devuelve la resta de ambos
  @param a: primer numero
  @param b: segundo numero
  @return: la resta de a y b
  """
  print(f"restando a: {a} con b: {b}")
  return a - b

@tool
def get_current_date() -> dict:
    """
    Obtener la fecha actual en el formato YYYY-MM-DD
    """
    return {"current_date": datetime.now().strftime("%Y-%m-%d")}

tools = [get_weather, sumar, multiplicar, restar, retrieve_context, get_current_date]

In [8]:
from langchain.agents import create_agent

system_msg = "Eres un asistente util"
agent = create_agent(
    model = llm_model,
    system_prompt = system_msg,
    tools = tools
)

In [10]:
pregunta = "cuales son las opciones de matriculas?"

# Use with chat models
messages = []
messages.append(HumanMessage(pregunta))

result = agent.invoke(
    {"messages": messages}
)

print(result["messages"][-1].content)

Las opciones de matrícula incluyen las siguientes modalidades de pago:

1. **Pago al Contado**: Los estudiantes pueden optar por pagar el total del programa al momento de la matrícula con un descuento del 15%. Este descuento se aplica sobre el precio regular y debe realizarse mediante transferencia bancaria o depósito directo.

2. **Financiamiento Interno**: Se ofrecen planes de financiamiento interno sin intereses hasta en 6 cuotas mensuales. El pago inicial corresponde al 30% del programa, y el saldo se divide en cuotas iguales. Se requiere una evaluación crediticia básica y la firma de un pagaré.

Si necesitas más información o detalles específicos, no dudes en preguntar.


Referencias:

https://docs.langchain.com/oss/python/langchain/rag#2-retrieval-and-generation
https://docs.langchain.com/oss/python/langchain/agents